# Data preparation of outage data from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Monthly availability for peakload technlogies and country for selected year - both on a generation and production unit level (avail_peakload_"+outagetype+"_"+year+"_monthly_entsoe.csv)

Settings in next window

In [1]:
#download files again (yes/no)?
download = "yes"

#set year for data creation
year = '2024'

#outage type GU/PU
outagetype = "GenerationUnits_15.1.A_B"  ## Leon, Generating Unit (GU) (GenerationUnits_15.1.A_B) vs. Production Unit (PU) (ProductionUnits_15.1.C_D) 

In [2]:
import pysftp
import sys
import os
import pandas as pd
import datetime as dt
import wget
import calendar

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
password = "3?mJ}V4us?!L}5E"                
username = "jsavelsberg@ethz.ch"                
port = '22'

In [4]:
dir_out = "../parsed_data/"

In [5]:
#technology definition
dict_agg_tech = {'Fossil Oil shale ': 'Oil', ##
                 'Fossil Gas': 'Gas', ##
                 'Fossil Brown coal/Lignite ': 'Lignite', ## 
                 'Hydro Water Reservoir ': 'Reservoir', ## 
                 'Hydro Pumped Storage ': 'Pump', ##
                 'Other ': 'Other', ##
                 'Coal-derived gas' : 'Gas',
                 'Fossil Peat' : 'Other', 
                 'Other renewable' : 'Other', 
                 'Waste' : 'Other',
                 'Fossil Hard coal ': 'HardCoal', ##
                 'Fossil Oil ': 'Oil', ## 
                 'Nuclear ': 'Nuclear', ##
                 'Wind Offshore ': 'WindOffshore', ## 
                 'Biomass ': 'Biomass', ## 
                 'Wind Onshore ': 'WindOnshore', ## 
                 'Solar' : 'Solar', ### NEW 
                 'Geothermal' : 'Geothermal', ### NEW 
                 'Hydro Run-of-river and poundage ': 'RunOfRiver'} ## 
# Leon: 
# Fossil Coal-derived gas, Fossil Peat, Solar, Geothermal, Other renewable, Waste 
# ??????? (die sind in ProductionType enthalten, aber nicht hier im Mapping) 
# das sollte zu NaNs führen > deswegen nehmen wir die neuen technologies mit auf 
# Solar und Geothermal als neue Kategorien 

In [6]:
print(list(dict_agg_tech.values())) # [neue Zuweisungen]

['Oil', 'Gas', 'Lignite', 'Reservoir', 'Pump', 'Other', 'Gas', 'Other', 'Other', 'Other', 'HardCoal', 'Oil', 'Nuclear', 'WindOffshore', 'Biomass', 'WindOnshore', 'Solar', 'Geothermal', 'RunOfRiver']


In [7]:
dict_agg_country = {
    'EE': 'EE', ## 
    'HU': 'HU', ## 
    'CZ': 'CZ', ## 
    'BE': 'BE', ## 
    'CH': 'CH', ## 
    'DE_TransnetBW': 'DE', ## Germany ?????? Doppeltzählung vermeiden ????????
    'DE_LU': 'DE', # NEW:  DE_LU
    'FR': 'FR', ## 
    'NO2': 'NO', ## 
    'NO': 'NO', ## 
    'DE_Amprion': 'DE', ## 
    'NO5': 'NO', ## 
    'IT-CSOUTH': 'IT', # NEW: Italy ?????? Doppeltzählung vermeiden ????????
    'IT-Calabria': 'IT', 
    'IT-NORTH': 'IT',
    'IT-CNORTH': 'IT',
    'IT-Sicily': 'IT',
    'IT-Sardinia': 'IT',
    'IT-SOUTH': 'IT',
    'IT': 'IT', ## NEW: Italy
    'SE2': 'SE', ## 
    'SE': 'SE', ## 
    'DK2': 'DK', ##
    'ES': 'ES', ## 
    'DK': 'DK', ## 
    'NO1': 'NO', ## 
    'SE1': 'SE', ## 
    'SE3': 'SE', ## 
    'RO': 'RO', ##
    'LV': 'LV',##
    'DE_TenneT_GER': 'DE', ## 
    'AL': 'AL', # ??????????????????????
    'PL': 'PL', ## 
    'GB': 'GB', ## 
    'FI': 'FI', ## 
    'LT': 'LT', ##
    'DK1': 'DK', ## 
    'NO3': 'NO',
    'NO4': 'NO', ## 
    'DE_50HzT': 'DE', ## 
    'SE4': 'SE', ## 
    'MK': 'MK', ## 
    'AT': 'AT', ## 
    'NL': 'NL', ## 
    'SK': 'SK',
    'BG' : 'BG', # NEW
    'IE_SEM':'IE', #NEW  Ireland ?????? Doppeltzählung vermeiden ????????
    'IE':'IE', # NEW 
    'NIE':'IE', #NEW
    'SI':'SI', #NEW
    'PT':'PT', #NEW
    'XK':'XK', #NEW
    'ME':'ME', #NEW
    'GE':'GE', #NEW
    'GR':'GR', #NEW
    'RS':'RS', #NEW
    'BA':'BA', #NEW   
} 

In [8]:
months = pd.DataFrame()
for m in range(1,13):
    months[m] = calendar.monthrange(int(year), m)
dayspermonth = months.loc[1]

In [9]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [10]:
# show list of all available folders (uncomment last line if needed)
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    files = sftp.listdir('/TP_export/')   
    print(files)

Connection succesfully established.
['AcceptedAggregatedOffers_17.1.D', 'ActivatedBalancingEnergy_17.1.E', 'ActualCapacitiesAndOutlookOnFrequencyRestorationReserveAndReplacementReserve_SOGL_188.3_188.4_189.2_189.3_r3', 'ActualGenerationOutputPerGenerationUnit_16.1.A_r2.1', 'ActualTotalLoad_6.1.A', 'AggregatedBalancingEnergyBids_12.3.E_r3', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D_r3', 'AggregatedGenerationPerType_16.1.B_C', 'AmountAndPricesPaidOfBalancingReservesUnderContract_17.1.B_C_r2', 'AmountOfBalancingReservesUnderContract_17.1.B', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r2', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r3', 'ChangesInActualAvailabilityOfConsumptionUnits_7.1.B', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructureReasons_10.1.C', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructure_10.1.C', 'ChangesToBidAvailability_IFs_mFRR9.9_aFRR9.6_9.8_r3', 'Com

In [11]:
#load file names from server
path_out = path+'UnavailabilityOf'+outagetype+'/'
path_out_local = path_local+'outages/'

# Leon: (Ordner sicherstellen)
os.makedirs(path_out_local, exist_ok=True)

with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_out)
    #print(files)
    if year != "":
        files = [i for i in files if year in i]

Connection succesfully established.


In [12]:
#download aggregated generation data (AggregatedGenerationPerType)
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_out+file,path_out_local+file)
            print('Successfully downloaded file '+file)

Successfully downloaded file 2024_01_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_02_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_03_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_04_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_05_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_06_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_07_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_08_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_09_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_10_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_11_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Successfully downloaded file 2024_12_UnavailabilityOfGenerationUnits_15.1.A_B.csv


In [13]:
#Leon
import chardet
# Eine der Datein testen
test_file = path_out_local + files[0]
print("Prüfe Datei:", test_file)

# Encoding automatisch erkennen
with open(test_file, 'rb') as f:
    result = chardet.detect(f.read(50000)) # nur ersten Teil lesen
print("Erkanntes Encoding:", result)

# Datei mit erkanntem Encoding laden
df_test = pd.read_csv(test_file, sep="\t", encoding=result['encoding'], nrows=5)
print("Spalten:", df_test.columns.tolist())



Prüfe Datei: ..\source_data/ENTSOE/outages/2024_01_UnavailabilityOfGenerationUnits_15.1.A_B.csv
Erkanntes Encoding: {'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': ''}
Spalten: ['StartOutage', 'EndOutage', 'StartTS', 'EndTS', 'TimeZone', 'MRID', 'Status', 'Type', 'AreaCode', 'AreaTypeCode', 'AreaName', 'MapCode', 'PowerResourceEIC', 'UnitName', 'ProductionType', 'InstalledCapacity', 'AvailableCapacity', 'Version', 'OldVersion', 'Reason', 'UpdateTime']


In [14]:
df_out_in = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_out_local+file,
                          decimal=".",encoding="UTF-8-SIG",sep="\t")   # Leon, updated encoding 
    #df_out_in = df_out_in.append(df_temp)
    df_out_in = pd.concat([df_out_in, df_temp], ignore_index=True) # Leon, in pandas 2.0 gibt es .append() nicht mehr als Methode

print(df_out_in['MapCode'].unique())    

['IT-Calabria' 'HU' 'CZ' 'GB' 'BE' 'DE_LU' 'CH' 'ES' 'FR' 'DE_TenneT_GER'
 'NL' 'AT' 'BG' 'IE_SEM' 'SK' 'IT' 'DE_Amprion' 'LT' 'PL' 'NIE' 'RO' 'NO'
 'IT-CSOUTH' 'IT-NORTH' 'NO2' 'IT-Sicily' 'SI' 'DK1' 'PT' 'DK' 'SE' 'NO5'
 'IT-Sardinia' 'IT-CNORTH' 'DE_50HzT' 'DE_TransnetBW' 'FI' 'IT-SOUTH'
 'NO4' 'SE2' 'EE' 'SE3' 'XK' 'IE' 'DK2' 'ME' 'GE' 'SE1' 'NO1' 'GR' 'RS'
 'BA' 'LV' 'SE4' 'MK']


In [15]:
print(df_out_in['ProductionType'].unique())    

['Hydro Run-of-river and poundage' 'Fossil Gas'
 'Fossil Brown coal/Lignite' 'Fossil Hard coal' 'Nuclear'
 'Hydro Pumped Storage' 'Fossil Coal-derived gas' 'Hydro Water Reservoir'
 'Other' 'Fossil Oil' 'Fossil Peat' 'Wind Onshore' 'Biomass' 'Solar'
 'Geothermal' 'Fossil Oil shale' 'Wind Offshore' 'Other renewable' 'Waste']


In [16]:
df_out_in['ProductionType'].isna().sum()

np.int64(0)

In [17]:
#combine files to one data frame
df_out_in = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_out_local+file,
                          decimal=".",encoding="UTF-8-SIG",sep="\t")   # Leon, updated encoding 
    #df_out_in = df_out_in.append(df_temp)
    df_out_in = pd.concat([df_out_in, df_temp], ignore_index=True) # Leon, in pandas 2.0 gibt es .append() nicht mehr als Methode

# Leon, ???????? Manuelle Def. von UnavailabilityValue (was früher direkt in der Datei enthalten war):
df_out_in['UnavailabilityValue'] = df_out_in['InstalledCapacity'] - df_out_in['AvailableCapacity'] 

df_out_in = df_out_in[(df_out_in.Status == "Active") 
                          & (df_out_in.Type == "Planned") 
                          #& (df_out_in.UnavailabilityValue > 0)
                          & (df_out_in.AreaTypeCode == "CTA")
                         ]
df_out_in["technology"] = df_out_in.ProductionType.map(dict_agg_tech) # TECH - MAPPING
df_out_in["country"] = df_out_in.MapCode.map(dict_agg_country) # COUNTRY - MAPPING
#we ignore timezones here as we are only interested in monthly values
df_out_in['duration'] = (pd.to_datetime(df_out_in['EndTS']) - pd.to_datetime(df_out_in['StartTS']))/ pd.Timedelta('1 hour') # tatsächliche Ausfallzeit in Stunden 
df_out_in['gen_reduction'] = df_out_in['UnavailabilityValue'] * df_out_in['duration']

# Leon, Spalte Month manuell erzeugen (was früher direkt in der Datei enthalten war): 
df_out_in['Month'] = pd.to_datetime(df_out_in['StartTS']).dt.month

df_out_in['dayspermonth'] = df_out_in.Month.map(dayspermonth) # MONTH - MAPPING 
    
if outagetype == "ProductionUnits_15.1.C_D": # Leon,  (="PU" = Production Unit)
    df_out_in['fullload'] = df_out_in['dayspermonth'] * 24 * df_out_in['InstalledCapacity']
    
if outagetype == "GenerationUnits_15.1.A_B": # Leon, (="GU" = Generating Unit)
    df_out_in['fullload'] = df_out_in['dayspermonth'] * 24 * df_out_in['InstalledCapacity'] # PROBLEM InstalledGenCapacity???????

#df_out_in.tail()
df_out_in["technology"].unique()

array(['Gas', nan, 'Other', 'Geothermal', 'Solar'], dtype=object)

In [18]:
# Leon, MAPPING - Checks: 
print(set(dict_agg_tech.values()) - set(df_out_in["technology"]))


{'WindOnshore', 'Oil', 'WindOffshore', 'RunOfRiver', 'Lignite', 'Pump', 'HardCoal', 'Biomass', 'Reservoir', 'Nuclear'}


In [19]:
if outagetype == "ProductionUnits_15.1.C_D":  # "PU"
    df_out = df_out_in.groupby(['Month','country','technology']).sum()[['InstalledCapacity','fullload','gen_reduction']]
    
if outagetype == "GenerationUnits_15.1.A_B":  #"GU"
    df_out = df_out_in.groupby(['Month','country','technology']).sum()[['InstalledCapacity','fullload','gen_reduction']] # PROBLEM InstalledGenCapacity

df_out['availability'] = (df_out['fullload'] - df_out['gen_reduction']) / df_out['fullload']
df_out.head()

InstalledCapacity    fullload  gen_reduction  \
Month country technology                                                 
1     AT      Gas                    5194.0   3864336.0   1.158290e+06   
      BE      Gas                    7448.8   5541907.2   2.972755e+05   
      BG      Gas                     420.0    312480.0   3.026100e+05   
      CZ      Gas                   42000.0  31248000.0   8.928680e+05   
      DE      Gas                   80958.4  60233049.6   5.577155e+06   

                          availability  
Month country technology                
1     AT      Gas             0.700262  
      BE      Gas             0.946359  
      BG      Gas             0.031586  
      CZ      Gas             0.971426  
      DE      Gas             0.907407

In [20]:
df_availability = df_out['availability'].reset_index().pivot_table(values='availability', index=['country','technology'], columns='Month')
df_availability = df_availability.fillna(1)
df_availability = df_availability.reset_index()[df_availability.reset_index().technology.isin(['Gas','Oil','HardCoal'])].set_index(['country','technology'])
df_availability = abs(df_availability)
df_availability[df_availability > 1] = 1
df_availability = pd.DataFrame(df_availability.stack()).rename(columns = {0:'avail'})
df_availability.head()

avail
country technology Month          
AT      Gas        1      0.700262
                   2      0.575143
                   3      0.374085
                   4      0.188551
                   5      0.104083

In [15]:
df_availability.to_csv(dir_out + "avail_peakload_"+outagetype+"_"+year+"_monthly_entsoe.csv", index=True)